# Getting Started with `snbb_atlas_pack`

This notebook demonstrates the core Python API for the **SNBB Atlas Pack** — a BIDS-compatible collection of 46 brain parcellation atlases covering subcortical, cortical, and combined spaces.

**Atlases included:**

| Family | Count | Space | Format |
|--------|------:|-------|--------|
| Tian Melbourne Subcortex (S1–S4) | 4 | MNI152NLin2009cAsym 1 mm | NIfTI |
| HCPex (cortex + subcortex) | 1 | MNI152NLin2009cAsym 1 mm | NIfTI |
| HCP-MMP 1.0 (surface) | 1 | fsLR 32k | GIFTI |
| Schaefer2018 + Tian (combined) | 40 | MNI152NLin2009cAsym 1 mm | NIfTI |

> **Prerequisites:** `uv sync` from the repo root to install dependencies.

In [ ]:
import snbb_atlas_pack as snbb
import pandas as pd
import nibabel as nib
import numpy as np

print(f"snbb_atlas_pack loaded — {len(snbb.list_atlases())} atlases available")

---

## 1. Discovering Available Atlases

`list_atlases()` returns a sorted list of all atlas IDs you can pass to `get_atlas()`.

In [ ]:
all_ids = snbb.list_atlases()
print(f"Total: {len(all_ids)} atlases\n")

# Group by family
families = {
    "Tian":    [a for a in all_ids if a.startswith("Tian")],
    "HCPex":   [a for a in all_ids if a.startswith("HCPex")],
    "HCPMMP":  [a for a in all_ids if a.startswith("HCPMMP")],
    "Schaefer": [a for a in all_ids if a.startswith("Schaefer")],
}

for family, ids in families.items():
    print(f"{family:12s} ({len(ids):2d}):  {ids[0]}  …  {ids[-1]}")

---

## 2. Tian Melbourne Subcortex Atlas

Four hierarchical scales of subcortical parcellation (bilateral structures: hippocampus, amygdala, thalamus, NAc, GP, putamen, caudate).

In [ ]:
atlas = snbb.get_atlas("TianS1")

print("AtlasResult fields")
print(f"  atlas_id : {atlas.atlas_id}")
print(f"  modality : {atlas.modality}")
print(f"  space    : {atlas.space}")
print(f"  maps     : {atlas.maps.name}")
print(f"  maps_R   : {atlas.maps_R}   ← None for volumetric")
print(f"  labels   : {len(atlas.labels)} regions × {len(atlas.labels.columns)} columns")

In [ ]:
# The labels table is a plain pandas DataFrame
atlas.labels

In [ ]:
# Region counts per Tian scale
print("Regions per Tian scale:")
for scale in ["TianS1", "TianS2", "TianS3", "TianS4"]:
    n = len(snbb.get_atlas(scale).labels)
    print(f"  {scale}: {n:2d} regions")

In [ ]:
# Explore label table: bilateral structure breakdown at scale S4
s4 = snbb.get_atlas("TianS4")

print("TianS4 — regions per structure:")
print(
    s4.labels.groupby(["structure", "hemisphere"])
    .size()
    .unstack(fill_value=0)
    .rename(columns={"L": "Left", "R": "Right"})
    .to_string()
)
print(f"\nMNI centre-of-gravity range  x: [{s4.labels.x_cog.min():.0f}, {s4.labels.x_cog.max():.0f}] mm")

---

## 3. HCPex — Cortex + Subcortex

426-region atlas: 360 HCP-MMP cortical areas + 66 subcortical structures (7 bilateral pairs). The label table carries RGB colours, lobe annotations, and volumetric statistics.

In [ ]:
hcpex = snbb.get_atlas("HCPex")
print(f"HCPex: {len(hcpex.labels)} regions")
print(f"Columns: {list(hcpex.labels.columns)}")
hcpex.labels.head(8)

In [ ]:
# Breakdown by lobe (cortical regions)
cortex = hcpex.labels[hcpex.labels["lobe"].notna()]
print("Regions per lobe:")
print(cortex["lobe"].value_counts().to_string())

print("\nLargest regions by volume (top 10):")
print(
    hcpex.labels.nlargest(10, "volume_mm3")[["label", "hemisphere", "lobe", "volume_mm3"]]
    .to_string(index=False)
)

In [ ]:
# RGB colours are stored per region — useful for consistent colouring in custom plots
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Aggregate to one representative colour per lobe
lobe_palette = (
    hcpex.labels.dropna(subset=["lobe"])
    .groupby("lobe")[["r", "g", "b"]]
    .mean()
    .astype(int)
)

fig, ax = plt.subplots(figsize=(7, 1.2))
ax.set_xlim(0, len(lobe_palette))
ax.set_ylim(0, 1)
ax.axis("off")
ax.set_title("HCPex lobe colours (mean RGB)", fontsize=10)

for i, (lobe, row) in enumerate(lobe_palette.iterrows()):
    colour = tuple(row[["r", "g", "b"]] / 255)
    ax.add_patch(mpatches.Rectangle((i, 0), 1, 1, color=colour))
    ax.text(i + 0.5, -0.15, lobe, ha="center", va="top", fontsize=8)

plt.tight_layout()
plt.show()

---

## 4. HCP-MMP 1.0 — Surface Atlas

360-region (180/hemisphere) cortical parcellation in **fsLR 32k surface space**. `get_atlas()` returns separate GIFTI paths for left and right hemispheres.

In [ ]:
hcpmmp = snbb.get_atlas("HCPMMP")

print(f"modality : {hcpmmp.modality}")
print(f"space    : {hcpmmp.space}")
print(f"maps     : {hcpmmp.maps.name}")
print(f"maps_R   : {hcpmmp.maps_R.name}")
print(f"labels   : {len(hcpmmp.labels)} regions")

In [ ]:
# Fetch a single hemisphere
lh_only = snbb.get_atlas("HCPMMP", hemi="L")
print(f"Left-only: maps={lh_only.maps.name}, maps_R={lh_only.maps_R}")

# Load the GIFTI with nibabel
gifti = nib.load(hcpmmp.maps)
vertex_labels = gifti.darrays[0].data
print(f"\nGIFTI vertex array shape: {vertex_labels.shape}  (one label per 32k vertex)")
print(f"Unique non-zero labels   : {len(np.unique(vertex_labels[vertex_labels > 0]))}")

In [ ]:
hcpmmp.labels.head(8)

In [ ]:
# Distribution across lobes and cortex types
print("Regions per lobe:")
print(hcpmmp.labels["lobe"].value_counts().to_string())

print("\nTop cortex types (top 8):")
print(hcpmmp.labels["cortex_type"].value_counts().head(8).to_string())

---

## 5. Schaefer2018 + Tian — Combined Atlases

40 combined atlases: every combination of **10 Schaefer cortical resolutions** (100–1000 regions, 7-network solution) with **4 Tian subcortical scales** (S1–S4). Labels include a `component` column (`"cortex"` / `"subcortex"`) and a `network` column for cortical regions.

In [ ]:
schaefer = snbb.get_atlas("Schaefer2018N400n7Tian2020S2")
print(f"{schaefer.atlas_id}")
print(f"  {len(schaefer.labels)} total regions")
print(f"  columns: {list(schaefer.labels.columns)}")
print()
print(schaefer.labels["component"].value_counts().to_string())

In [ ]:
# Cortical regions carry 7-network assignments
cortex_rows = schaefer.labels[schaefer.labels["component"] == "cortex"]
print("Cortical regions per network:")
print(cortex_rows["network"].value_counts().to_string())

print("\nSubcortical regions (TianS2):")
print(
    schaefer.labels[schaefer.labels["component"] == "subcortex"][["label", "hemisphere"]]
    .to_string(index=False)
)

In [ ]:
# All 40 Schaefer-Tian variants — total region counts
records = []
for n in [100, 200, 400, 600, 1000]:
    for s in [1, 2, 3, 4]:
        atlas_id = f"Schaefer2018N{n}n7Tian2020S{s}"
        labels = snbb.get_atlas(atlas_id).labels
        records.append(
            {"N": n, "S": s, "cortex": (labels["component"] == "cortex").sum(),
             "subcortex": (labels["component"] == "subcortex").sum(),
             "total": len(labels)}
        )

summary = pd.DataFrame(records)
print(summary.pivot(index="N", columns="S", values="total").to_string())

---

## 6. Loading Images with nibabel

The `maps` attribute is a `pathlib.Path` — you load the image yourself.

In [ ]:
# Volumetric NIfTI
atlas = snbb.get_atlas("TianS1")
img = nib.load(atlas.maps)

print(f"Image shape  : {img.shape}")
print(f"Voxel size   : {img.header.get_zooms()} mm")
print(f"Affine (first row): {img.affine[0]}")

data = img.get_fdata()
unique_labels = np.unique(data[data > 0]).astype(int)
print(f"Non-zero labels: {unique_labels}")

In [ ]:
# Quick coronal slice through the subcortex
import matplotlib.pyplot as plt

data = img.get_fdata()
slice_y = data.shape[1] // 2   # coronal mid-slice

fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(data[:, slice_y, :].T, origin="lower", cmap="tab20", interpolation="nearest")
ax.set_title(f"TianS1 — coronal slice y={slice_y}")
ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Compute voxel volume per region and compare to TSV x/y/z centres
voxel_vol_mm3 = np.prod(img.header.get_zooms())  # 1.0 mm³ at 1 mm isotropic
data_int = data.astype(int)
voxel_counts = {lbl: np.sum(data_int == lbl) for lbl in unique_labels}

region_vols = (
    atlas.labels
    .assign(volume_mm3=lambda df: df["index"].map(
        {lbl: cnt * voxel_vol_mm3 for lbl, cnt in voxel_counts.items()}
    ))
    [["label", "hemisphere", "structure", "volume_mm3"]]
)
print(region_vols.to_string(index=False))

---

## 7. Error Handling

The API raises descriptive errors for invalid inputs.

In [ ]:
# Unknown atlas ID
try:
    snbb.get_atlas("DoesNotExist")
except KeyError as e:
    print(f"KeyError: {e}")

# hemi= on a volumetric atlas
try:
    snbb.get_atlas("TianS1", hemi="L")
except ValueError as e:
    print(f"ValueError: {e}")

# Mesh not yet built
try:
    snbb.get_mesh("TianS1")  # raises FileNotFoundError if derivatives not populated
    print("Mesh found at:", snbb.get_mesh("TianS1"))
except FileNotFoundError as e:
    print(f"FileNotFoundError: {e}")

---

## Summary

| Function | Returns | Notes |
|----------|---------|-------|
| `snbb.list_atlases()` | `list[str]` | Sorted list of all 46 atlas IDs |
| `snbb.get_atlas(id)` | `AtlasResult` | `maps` (Path) + `labels` (DataFrame) |
| `snbb.get_atlas(id, hemi="L")` | `AtlasResult` | Surface atlases only |
| `snbb.get_mesh(id)` | `Path` | Prebuilt yabplot mesh directory |
| `snbb.list_meshes()` | `dict` | `{atlas_id: [components]}` |
| `snbb.build_meshes(id)` | `None` | Build/rebuild yabplot mesh cache |

Continue to **`02_visualization.ipynb`** for atlas figure display and custom data overlays.